# Tree embedding / clustering with the current matcher stack

This notebook re-implements the old clustering workflow using the current sampler and matcher code.

The overall pipeline is still:

1. Compute matcher-based tree similarities (or approximate surrogates).
2. Normalize them.
3. Run UMAP.
4. Run HDBSCAN.

The main differences are:

- you can now choose among **full pairwise**, **landmark**, and **approximate kNN-candidate** scoring modes;
- you can use either the **generic matcher** or the newer **fast matcher**;
- the old `embed_and_cluster_sns(...)` wrapper still exists for convenience, but the main API is now class-based via `TreeClusterEmbedder`.


In [ ]:
import os
import sys
from pathlib import Path
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from embedder.tree_cluster import TreeClusterEmbedder, embed_and_cluster_sns, extract_alpha_seqs_by_cluster
from embedder.utils import (
    pack_graphs_by_class,
    make_inverse_frequency_token_weights,
    plot_embedding_result,
    result_overview_frame,
)
from path_matcher.planted_path_sampler import (
    ToyModelConfig,
    make_default_tree_samplers,
    sample_toy_model,
)

print('Repo root:', REPO_ROOT)

## Sample a synthetic dataset with the current sampler

This replaces the old `gw_sampler` workflow. We sample trees from the toy model in the current codebase, then optionally repack them into the older nested `some_trees` structure so the compatibility wrapper is easy to demo.


In [ ]:
cfg = ToyModelConfig(
    n_classes=4,
    n_per_class=[18, 18, 18, 18],
    planted_seq_len=10,
    alphabet_size=10,
    p_obs=0.90,
    dirichlet_alpha=1.0,
    seed=SEED,
)

samplers = make_default_tree_samplers(
    max_depth=11,
    lam=1.7,
    max_nodes=180,
    min_nodes=160,
    require_full_depth=True,
)

graphs, gt_labels, class_sequences = sample_toy_model(
    cfg,
    tree_path_sampler=samplers['gw_bfs'],
)

# Optional: repack into the older nested structure.
some_trees = pack_graphs_by_class(graphs, gt_labels, class_sequences=class_sequences)

print('n graphs =', len(graphs))
print('class counts =', np.bincount(np.asarray(gt_labels, dtype=int)))
print('example class sequence =', class_sequences[0])

## 1. Compatibility wrapper: full pairwise similarities

This is the closest analogue of the old `gw_clustering.embed_and_cluster_sns(...)` call.

Here I use the **fast exact matcher** in equality mode, which is a good default whenever the vertex-pair score is just weighted exact label equality.


In [ ]:
%%time
matches_full, embedding_full, tree_list_full, gt_full, pred_full = embed_and_cluster_sns(
    some_trees,
    similarity_mode='full',
    matcher_kind='fast',
    matcher_kwargs=dict(mode='equality'),
    normalizer='jaccard',
    phi_name='label',
    diag_source='row_max',
    random_state=SEED,
    title='Full pairwise similarities (fast equality matcher)',
)

print('ARI / NMI =',
      pd.Series({
          'ari': __import__('sklearn.metrics').metrics.adjusted_rand_score(gt_full, pred_full),
          'nmi': __import__('sklearn.metrics').metrics.normalized_mutual_info_score(gt_full, pred_full),
      }).round(3).to_dict())

## 2. Class-based API: weighted similarities

This mirrors the old rare-label experiment, but now the fast matcher can consume a token-weight dictionary directly.

A simple empirical choice is inverse token frequency over the sampled dataset.


In [ ]:
token_weights = make_inverse_frequency_token_weights(graphs, phi_name='label', power=1.0)
list(token_weights.items())[:5]

In [ ]:
%%time
full_weighted = TreeClusterEmbedder(
    similarity_mode='full',
    matcher_kind='fast',
    matcher_kwargs=dict(mode='equality', token_weights=token_weights),
    normalizer='jaccard',
    phi_name='label',
    diag_source='row_max',
    random_state=SEED,
)
res_full_weighted = full_weighted.fit_predict(graphs)
plot_embedding_result(res_full_weighted, title='Full pairwise, weighted exact-label similarity')
plt.show()

## 3. Landmark mode

Instead of computing every pairwise score, compute scores only to a moderate number of landmarks.

The resulting **tree-to-landmark score vectors** become the input features for UMAP. This is often the cleanest speedup when the matcher itself is the bottleneck.


In [ ]:
%%time
landmark_embedder = TreeClusterEmbedder(
    similarity_mode='landmarks',
    matcher_kind='fast',
    matcher_kwargs=dict(mode='equality'),
    normalizer='jaccard',
    phi_name='label',
    n_landmarks=20,
    landmark_mode='spread',
    feature_metric='euclidean',
    random_state=SEED,
)
res_landmarks = landmark_embedder.fit_predict(graphs)
plot_embedding_result(res_landmarks, title='Landmark features + UMAP + HDBSCAN')
plt.show()
print('Landmarks used:', res_landmarks.landmark_indices.tolist())

## 4. Approximate kNN-candidate scoring

Here we first build a **cheap tree sketch** (hashed token counts), find candidate neighbours in sketch space, and only then compute exact matcher scores on those candidate pairs.

This is the most natural place to plug in NN-descent style acceleration. If `pynndescent` is installed, the module will use it automatically; otherwise it falls back to `sklearn.neighbors`.


In [ ]:
%%time
approx_embedder = TreeClusterEmbedder(
    similarity_mode='approx_knn',
    matcher_kind='fast',
    matcher_kwargs=dict(mode='equality'),
    normalizer='jaccard',
    phi_name='label',
    candidate_top_k=12,
    candidate_backend='auto',
    candidate_metric='cosine',
    sketch_hash_dim=256,
    random_state=SEED,
)
res_approx = approx_embedder.fit_predict(graphs)
plot_embedding_result(res_approx, title='Approximate candidate graph + exact rescoring')
plt.show()
print('Number of candidate pairs scored exactly:', len(res_approx.candidate_pairs))

## 5. Compare the three regimes


In [ ]:
overview = result_overview_frame({
    'full_unweighted': TreeClusterEmbedder(
        similarity_mode='full',
        matcher_kind='fast',
        matcher_kwargs=dict(mode='equality'),
        normalizer='jaccard',
        phi_name='label',
        diag_source='row_max',
        random_state=SEED,
    ).fit_predict(graphs),
    'full_weighted': res_full_weighted,
    'landmarks': res_landmarks,
    'approx_knn': res_approx,
})
display(overview.round(3))

## 6. Extract matched sequences inside clusters

This is the analogue of the old workflow where matched paths were converted into noisy sequence observations.


In [ ]:
cluster_seqs = extract_alpha_seqs_by_cluster(
    matches_full,
    gt_full,
    tree_list_full,
    phi_name='label',
)

for c, seqs in cluster_seqs.items():
    print(f'class {c}: {len(seqs)} extracted noisy sequences')
    if seqs:
        print('  first example =', seqs[0])
        break

## Notes

A few good defaults in practice:

- **Full pairwise + fast matcher** when the dataset is still small enough that `n^2` scoring is acceptable.
- **Landmarks** when you mainly want a speedup and can tolerate a low-rank approximation to the similarity geometry.
- **Approximate kNN candidates** when the dataset is large and you mostly care about preserving the strongest local neighbours before UMAP/HDBSCAN.

For more flexible vertex weights (custom `w(label_u, label_v)`), switch `matcher_kind='generic'` and pass the usual `TreePathMatcher` kwargs in `matcher_kwargs`.
